# Orchestrator-Subagent — CrewAI

**Pattern:** Specialists with focused tools → orchestrator assembles

```
highlights_specialist → task 1
logistics_specialist  → task 2
itinerary_writer      → task 3 (context=[t1,t2])
package_formatter     → task 4 (context=[t1,t2,t3]) → TripPackage
```

Each specialist has one tool. The final formatter has `context=[all_tasks]`.

In [ ]:
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
gemini = LLM(model="gemini/gemini-2.0-flash", temperature=0)
print("✓ Ready")

In [ ]:
@tool("City Highlights Lookup")
def get_highlights(city: str) -> str:
    """Get top tourist highlights. Input: city name."""
    data = {"tokyo": "Shibuya, Senso-ji, Tsukiji, Mt Fuji", "paris": "Eiffel, Louvre, Notre Dame", "bangalore": "Lalbagh, Nandi Hills"}
    return data.get(city.lower(), f"No data for '{city}'.")

@tool("City Logistics Lookup")
def get_logistics(city: str) -> str:
    """Get travel logistics. Input: city name."""
    data = {"tokyo": "3h from SFO. Shinkansen. Best: Mar-May.", "paris": "11h from NYC. Metro. Best: Apr-Jun.", "bangalore": "9h from Dubai. Ola. Best: Oct-Feb."}
    return data.get(city.lower(), f"No data for '{city}'.")

class TripPackage(BaseModel):
    city: str
    highlights_summary: str
    logistics_summary: str
    itinerary: str
    package_overview: str

print("Tools + schema ready")

In [ ]:
highlights_spec = Agent(role="Highlights Researcher", goal="Find top attractions.",
    backstory="Expert travel guide.", tools=[get_highlights], llm=gemini, verbose=False)
logistics_spec  = Agent(role="Logistics Planner", goal="Find travel logistics.",
    backstory="Operations expert.", tools=[get_logistics], llm=gemini, verbose=False)
itinerary_writer = Agent(role="Itinerary Writer", goal="Create 3-day itinerary.",
    backstory="Craft detailed itineraries.", tools=[], llm=gemini, verbose=False)
formatter_agent  = Agent(role="Package Editor", goal="Assemble trip package.",
    backstory="Polish trip packages.", tools=[], llm=gemini, verbose=False)
print("4 specialist agents")

In [ ]:
city = "Tokyo"
t1 = Task(description=f"Research highlights for {city} with get_highlights.", expected_output="Top highlights.", agent=highlights_spec)
t2 = Task(description=f"Research logistics for {city} with get_logistics.", expected_output="Practical logistics.", agent=logistics_spec)
t3 = Task(description=f"Create 3-day itinerary for {city}.", expected_output="3-day itinerary.", agent=itinerary_writer, context=[t1,t2])
t4 = Task(description=f"Assemble TripPackage for {city}.", expected_output="Complete TripPackage.",
    agent=formatter_agent, context=[t1,t2,t3], output_pydantic=TripPackage)

result = Crew(agents=[highlights_spec,logistics_spec,itinerary_writer,formatter_agent],
    tasks=[t1,t2,t3,t4], process=Process.sequential, verbose=False).kickoff()
pkg: TripPackage = result.pydantic
if pkg:
    print(f"### {pkg.city}\nHighlights: {pkg.highlights_summary[:80]}...\nOverview: {pkg.package_overview[:100]}...")
else:
    print(result.raw[:300])

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Each specialist has ONE tool | True specialization — each agent knows exactly one domain |
| `context=[all_tasks]` on formatter | Final assembler sees all outputs |
| `output_pydantic=TripPackage` | Structured output from multi-stage pipeline |

### Next: [ADK Orchestrator →](../ADK/orchestrator.ipynb)